# Introdução à Programação(BIA)
## Igor Jesus da Silva
### Matrícula: 202603053
#### E-mail institucional: igorjesus@discente.ufg.br

In [ ]:
#CADASTRO DOS ITENS
itens = []  # cria uma lista vazia que armazenará todos os itens cadastrados

qtd_itens = int(input("Quantos itens deseja cadastrar? "))  # solicita ao usuário a quantidade de itens a cadastrar e converte para inteiro

for i in range(qtd_itens):  # loop que será executado para cada item a ser cadastrado
    print(f"\nCadastro do item {i+1}")  # exibe o número do item atual (começando em 1)

    descricao = input("Descrição: ")  # solicita a descrição (nome) do item
    unidade = input("Unidade de medida: ")  # solicita a unidade de medida do item
    saldo_inicial = float(input("Saldo inicial: "))  # solicita o saldo inicial e converte para float
    consumo_diario = float(input("Consumo médio diário: "))  # solicita o consumo médio diário
    lead_time = int(input("Tempo de reposição (dias): "))  # solicita o tempo de reposição em dias

    # cada item será representado por um dicionário
    item = {
        "descricao": descricao,  # armazena a descrição do item
        "unidade": unidade,  # armazena a unidade de medida
        "saldo_inicial": saldo_inicial,  # armazena o saldo inicial
        "saldo_atual": saldo_inicial,  # inicializa o saldo atual com o saldo inicial
        "consumo_diario": consumo_diario,  # armazena o consumo médio diário
        "lead_time": lead_time,  # armazena o tempo de reposição
        "lotes": [(saldo_inicial, 0)],  # cria uma lista de lotes com um lote inicial (quantidade, valor unitário)
        "total_saidas": 0  # inicializa o total de saídas (giro) como zero
    }

    itens.append(item)  # adiciona o dicionário do item à lista de itens


# 2) NÚMERO DE MOVIMENTAÇÕES
qtd_mov = int(input("\nQuantas movimentações deseja registrar? "))  # solicita o número de movimentações e converte para inteiro

# 3) REGISTRO DAS MOVIMENTAÇÕES
for i in range(qtd_mov):  # loop que será executado para cada movimentação
    print(f"\nMovimentação {i+1}")  # exibe o número da movimentação atual

    nome_item = input("Item: ")  # solicita o nome do item a ser movimentado
    tipo = input("Tipo (ENTRADA/SAIDA): ").upper()  # solicita o tipo e converte para maiúsculas
    quantidade = float(input("Quantidade: "))  # solicita a quantidade movimentada
    valor = float(input("Valor unitário: "))  # solicita o valor unitário do item

    # procurar o item na lista
    item_encontrado = None  # inicializa como None (caso não encontre)
    for item in itens:  # percorre todos os itens cadastrados
        if item["descricao"] == nome_item:  # verifica se a descrição corresponde ao nome informado
            item_encontrado = item  # armazena o item encontrado
            break  # interrompe o loop assim que encontra o item

    # se item não existir
    if item_encontrado is None:  # verifica se não encontrou o item
        print("Erro: item não encontrado.")  # exibe mensagem de erro
        continue  # pula para a próxima movimentação

    #ENTRADA
    if tipo == "ENTRADA":  # verifica se a movimentação é uma entrada
        item_encontrado["saldo_atual"] += quantidade  # adiciona a quantidade ao saldo atual
        item_encontrado["lotes"].append((quantidade, valor))  # adiciona um novo lote (quantidade, valor)

    #SAÍDA
    elif tipo == "SAIDA":  # verifica se a movimentação é uma saída

        # verifica se vai ficar negativo
        if quantidade > item_encontrado["saldo_atual"]:  # verifica se a saída é maior que o saldo disponível
            print("Erro: saída maior que o saldo disponível. Operação cancelada.")  # mensagem de erro
            continue  # não executa a saída e passa para a próxima movimentação

        item_encontrado["saldo_atual"] -= quantidade  # subtrai a quantidade do saldo atual
        item_encontrado["total_saidas"] += quantidade  # soma a quantidade ao total de saídas (giro)

        # lógica PEPS
        restante = quantidade  # variável que controla quanto ainda precisa ser retirado dos lotes

        while restante > 0 and item_encontrado["lotes"]:  # executa enquanto ainda houver quantidade a retirar e existirem lotes
            lote_qtd, lote_valor = item_encontrado["lotes"][0]  # pega o primeiro lote da lista (mais antigo)

            if lote_qtd <= restante:  # verifica se o lote inteiro será consumido
                restante -= lote_qtd  # diminui o restante pela quantidade do lote
                item_encontrado["lotes"].pop(0)  # remove o lote consumido da lista
            else:
                item_encontrado["lotes"][0] = (lote_qtd - restante, lote_valor)  # atualiza o lote com a quantidade restante
                restante = 0  # zera o restante, pois já foi totalmente atendido

    else:
        print("Tipo inválido!")  # mensagem caso o tipo não seja ENTRADA nem SAIDA



# 4) RELATÓRIO FINAL
maior_giro = 0  # variável para armazenar o maior valor de saídas
item_maior_giro = None  # variável para armazenar o nome do item com maior giro

for item in itens:  # percorre todos os itens cadastrados

    # saldo atual
    saldo = item["saldo_atual"]  # armazena o saldo atual do item

    # cálculo do valor total do estoqu
    valor_total = 0  # inicializa o valor total
    for qtd, val in item["lotes"]:  # percorre todos os lotes do item
        valor_total += qtd * val  # soma o valor total (quantidade × valor unitário)

    # ponto de ressuprimento
    ponto_ressuprimento = item["consumo_diario"] * item["lead_time"]  # calcula o ponto de reposição

    # situação do estoque
    if saldo < ponto_ressuprimento:  # verifica se o saldo está abaixo do ponto de reposição
        status = "PRECISA COMPRAR"  # define status de compra necessária
    else:
        status = "OK"  # estoque está suficiente

    # comparação saldo inicial x atual
    if saldo > item["saldo_inicial"]:  # verifica se o saldo aumentou
        comparacao = "CRESCEU"
    elif saldo < item["saldo_inicial"]:  # verifica se o saldo diminuiu
        comparacao = "DIMINUIU"
    else:
        comparacao = "ESTÁVEL"  #saldo nao mudou

    # imprime informações
    print(f"\nItem: {item['descricao']}")  # imprime a descrição do item
    print(f"Saldo atual: {saldo}")  # imprime o saldo atual
    print(f"Valor total em estoque: {valor_total:.2f}")  # imprime o valor total formatado com 2 casas decimais
    print(f"Ponto de ressuprimento: {ponto_ressuprimento}")  # imprime o ponto de reposição
    print(f"Status: {status}")  # imprime o status do estoque
    print(f"Situação do estoque: {comparacao}")  # imprime a comparação com o saldo inicial

    # verifica maior giro
    if item["total_saidas"] > maior_giro:  # verifica se este item tem maior giro que o atual
        maior_giro = item["total_saidas"]  # atualiza o maior giro
        item_maior_giro = item["descricao"]  # armazena o nome do item com maior giro

# mostra item com maior giro
print("\nItem com maior giro:", item_maior_giro)  # imprime o item que teve maior saída

Quantos itens deseja cadastrar? 1

Cadastro do item 1
Descrição: Pneu
Unidade de medida: Unidade
Saldo inicial: 30
Consumo médio diário: 4
Tempo de reposição (dias): 7

Quantas movimentações deseja registrar? 1

Movimentação 1
Item: Pneu
Tipo (ENTRADA/SAIDA): Entrada
Quantidade: 1
Valor unitário: 100

Item: Pneu
Saldo atual: 31.0
Valor total em estoque: 100.00
Ponto de ressuprimento: 28.0
Status: OK
Situação do estoque: CRESCEU

Item com maior giro: None
